This notebook is for modeling and evaluating technique classification from the SemEval dataset. Given a span predicted to be propaganda, this model will identify which propaganda techniques, if any, were used.

Initially, we developed a **Hybrid RoBERTa Model** to incorporate manual linguistic features alongside deep learning embeddings. The goal was to determine if explicit metadata could assist the model in identifying complex propaganda techniques.

**Linguistic Features Added:**
1. **Sentiment Analysis:** Polarity and subjectivity scores (via TextBlob).
2. **Punctuation Density:** Count of exclamation marks, question marks, and CAPS, which are often markers of emotional "Loaded Language."
3. **Lexical Diversity:** Type-Token Ratio (TTR) to measure the complexity and variety of the vocabulary used in the span.

**Ablation Study Results:**
An ablation study was performed by comparing the Hybrid model's performance against a "Text-Only" baseline (where manual features were zeroed out).
* **Hybrid Micro-F1:** 0.1001
* **Text-Only Micro-F1:** 0.0987
* **Total Feature Lift:** +0.0014 (< 0.2%)

**Conclusion:**
The manual features provided a negligible lift of only **0.14%**. This indicates that the RoBERTa backbone is already capturing these linguistic nuances internally through its attention mechanisms.

**Decision:** To prioritize **model replicability** and **pipeline stability**, we have transitioned to a standard `AutoModelForSequenceClassification`. This allows for:
* **Native Hugging Face Support:** Automatic generation of `config.json` and `id2label` mappings.
* **Portability:** The model can be loaded in one line without requiring custom Python class definitions in downstream notebooks.

In [11]:
import pandas as pd
import numpy as np
import torch
import os
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit
from transformers.utils.notebook import NotebookProgressCallback
from safetensors.torch import load_file
import gdown
import zipfile
import shutil

In [12]:
#Define global paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models"
TC_MODEL_PATH = MODEL_DIR / "semeval_roberta_classifier"

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [4]:
#Download models if not already
def setup_models(file_id, target_path):
    target_path = Path(target_path).resolve()
    zip_temp = target_path.with_suffix(".zip")

    #Check if files already exist in the correct spot
    if (target_path / "model.safetensors").exists() or (target_path / "pytorch_model.bin").exists():
        print(f"Model weights detected locally at {target_path}")
        return True

    print(f"Model not found. Preparing {target_path}...")

    #Ensure the specific sub-folder exists
    target_path.mkdir(exist_ok=True, parents=True)

    url = f'https://drive.google.com/uc?id={file_id}'

    try:
        #1. Download the zip
        gdown.download(url, str(zip_temp), quiet=False)

        #2. Extract to a temporary location
        temp_extract = target_path / "temp_extraction"
        if temp_extract.exists(): shutil.rmtree(temp_extract)
        temp_extract.mkdir(parents=True)

        print("Unzipping and cleaning up structure...")
        with zipfile.ZipFile(zip_temp, 'r') as zip_ref:
            members = [m for m in zip_ref.namelist() if "__MACOSX" not in m]
            zip_ref.extractall(temp_extract, members=members)

        #3. Move files from temp_extract into target_path
        for root, dirs, files in os.walk(temp_extract):
            for file in files:
                src_file = Path(root) / file
                dest_file = target_path / file
                shutil.move(str(src_file), str(dest_file))

        #4. Final Cleanup
        shutil.rmtree(temp_extract)
        if zip_temp.exists():
            os.remove(zip_temp)

        print(f"Model files are now in: {target_path}")
        return True

    except Exception as e:
        print(f"Error during setup: {e}")
        if zip_temp.exists(): os.remove(zip_temp)
        return False

#Identify if model exists
model_already_trained = setup_models('1wZYbzVELfSz53gfPAZmy0QFqDbF2vvHb', MODEL_DIR)

Model weights detected locally at /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models


In [5]:
#Load technique classification data
df_tc = pd.read_csv(DATA_DIR / "semeval_tc_cleaned.csv")
df_tc.head()

,article_id,text_content,span_text,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,Bandwagon,Black-and-White_Fallacy,...,Loaded_Language,Minimisation,Name_Calling,Red_Herring,Reductio_ad_hitlerum,Repetition,Slogans,Straw_Men,Thought-terminating_Cliches,Whataboutism
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,0.250000,0,1.000000,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,111111111,Next plague outbreak in Madagascar could be 's...,"a very, very different",0.000000,0,1.000000,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,111111111,Next plague outbreak in Madagascar could be 's...,He also pointed to the presence of the pneumon...,0.483333,0,0.863636,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,111111111,Next plague outbreak in Madagascar could be 's...,but warned that the danger was not over,0.000000,0,1.000000,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
#Split the data by article rather than by span to avoid any data leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_tc, groups=df_tc['article_id']))
train_df = df_tc.iloc[train_idx].reset_index(drop=True)
test_df = df_tc.iloc[test_idx].reset_index(drop=True)

#Verify the split
print(f"Total spans: {len(df_tc)}, total articles: {df_tc['article_id'].nunique()}")
print(f"Train spans: {len(train_df)} ({train_df['article_id'].nunique()} articles)")
print(f"Test spans:  {len(test_df)} ({test_df['article_id'].nunique()} articles)")

#Check for leakage (should be 0)
overlap = set(train_df['article_id']).intersection(set(test_df['article_id']))
print(f"Number of overlapping articles: {len(overlap)}")

Total spans: 7587, total articles: 357
Train spans: 5856 (285 articles)
Test spans:  1731 (72 articles)
Number of overlapping articles: 0


In [7]:
#Identify feature columns versus input versus output
feature_cols = ['sentiment', 'punct_count', 'lexical_diversity']
label_cols = [c for c in train_df.columns if c not in (['article_id', 'text_content', 'span_text'] + feature_cols)]

In [8]:
#Put input text data in a format Hugging Face knows how to use
def preprocess_function(examples):
    # Standard RoBERTa tokenization
    encoding = tokenizer(examples["span_text"], padding="max_length", truncation=True, max_length=128)

    # Format labels as a float list for Multi-Label Classification
    labels_batch = {col: examples[col] for col in label_cols}
    labels_matrix = np.column_stack([labels_batch[col] for col in label_cols]).astype(float)
    encoding["labels"] = labels_matrix.tolist()

    return encoding

tokenized_train = Dataset.from_pandas(train_df).map(preprocess_function, batched=True)
tokenized_test = Dataset.from_pandas(test_df).map(preprocess_function, batched=True)

Map:   0%|          | 0/5856 [00:00<?, ? examples/s]

Map:   0%|          | 0/1731 [00:00<?, ? examples/s]

In [9]:
#Create the config so the model knows the names of the techniques
config = AutoConfig.from_pretrained(
    "roberta-base",
    num_labels=len(label_cols),
    id2label={i: label for i, label in enumerate(label_cols)},
    label2id={label: i for i, label in enumerate(label_cols)},
    problem_type="multi_label_classification"
)

#Load the standard model
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    config=config
).to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# If model already exists, load those weights
if model_already_trained:
    safe_path = TC_MODEL_PATH / "model.safetensors"
    bin_path = TC_MODEL_PATH / "pytorch_model.bin"

    state_dict = None

    if safe_path.exists():
        state_dict = load_file(safe_path, device="cpu")
    elif bin_path.exists():
        state_dict = torch.load(bin_path, map_location=torch.device('cpu'))

    if state_dict:
        # --- NEW: Fix the beta/gamma vs bias/weight mismatch ---
        new_state_dict = {}
        for key, value in state_dict.items():
            new_key = key.replace(".beta", ".bias").replace(".gamma", ".weight")
            new_state_dict[new_key] = value

        # Load the corrected state dict
        model.load_state_dict(new_state_dict, strict=True)
        print("Weights successfully renamed and injected into the model.")
    else:
        print(f"Error: No weights found. Check your paths!")

Weights successfully renamed and injected into the model.


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    #Apply sigmoid to get probabilities for multi-label
    probs = torch.sigmoid(torch.from_numpy(logits)).numpy()
    #Right now threshold is 0.5, but we can increase it to improve precision or lower it to improve recall
    predictions = (probs > 0.5).astype(int)

    return {
        "f1": f1_score(labels, predictions, average="micro"),
        "precision": precision_score(labels, predictions, average="micro"),
        "recall": recall_score(labels, predictions, average="micro"),
    }

In [16]:
training_args = TrainingArguments(
    output_dir=TC_MODEL_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2.47e-05,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.083,
    load_best_model_at_end=True
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [18]:
#Train only if we didn't load a local model
if not model_already_trained:
    print("Starting training process...")
    trainer.train()
    trainer.save_model(MODEL_DIR)
    tokenizer.save_pretrained(os.fspath(TC_MODEL_PATH))
    print(f"Model trained and saved to {MODEL_DIR}")
else:
    print("Model loaded from disk. Skipping training.")

Model loaded from disk. Skipping training.


In [19]:
#Evaluate performance on the test dataset
trainer.remove_callback(NotebookProgressCallback)
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print("\n" + "="*30)
print("FINAL MODEL PERFORMANCE")
print(f"Recall:    {test_results['eval_recall']:.4f}")
print(f"Precision: {test_results['eval_precision']:.4f}")
print(f"F1 Score:  {test_results['eval_f1']:.4f}")
print("="*30)

/Users/frankiepike/ds_env/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



FINAL MODEL PERFORMANCE
Recall:    0.5708
Precision: 0.7336
F1 Score:  0.6420
